# Best PainNAS architecture: single-target LOSO

This notebook trains the block-1 NAS winner **from scratch** for exactly one held-out BioVid subject. The target is excluded from training, validation, and normalization, and is evaluated once after fitting. Outputs are stored on Google Drive so the run can be resumed.

> The block-1 search directly excluded only its recorded outer-block subjects. Other targets require an explicit exploratory leakage override. Moreover, choosing block 1 after comparing all five winners is a post-search choice, so report this experiment as exploratory unless that decision was pre-specified.

## 1. Mount Drive and check out the repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys

REPO_URL = 'https://github.com/hhihn/FewShotPainAdaptation.git'
PROJECT_DIR = Path('/content/FewShotPainAdaptation')
BRANCH_NAME = 'painnas'

if not PROJECT_DIR.exists():
    !git clone -b $BRANCH_NAME $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull --ff-only
%cd $PROJECT_DIR

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
assert (PROJECT_DIR / 'scripts/run_painnas_best_single_loso.py').is_file()

## 2. Install dependencies

In [ ]:
!pip -q install -U pip
!pip -q install -r $PROJECT_DIR/painnas/requirements-colab.txt

## 3. Stage BioVid on the Colab SSD

The staging helper looks for the BioVid archive under `MyDrive/PainData`, copies/extracts it locally, and falls back to an already extracted Drive dataset.

In [ ]:
from data_loaders.dataset_staging import stage_predefined_dataset_from_archive

DRIVE_DATA_DIR = Path('/content/drive/MyDrive/PainData')
LOCAL_DATA_DIR = Path('/content/PainData')
BIOVID_ROOT = stage_predefined_dataset_from_archive(
    'biovid_part_a',
    drive_data_dir=DRIVE_DATA_DIR,
    local_data_dir=LOCAL_DATA_DIR,
    local_archive_dir=Path('/content'),
)
DATA_DIR = LOCAL_DATA_DIR
print('BioVid root:', BIOVID_ROOT)

## 4. Verify the GPU and configure reproducibility

In [ ]:
import random
import numpy as np
import tensorflow as tf

GPUS = tf.config.list_physical_devices('GPU')
assert GPUS, 'Select Runtime > Change runtime type > GPU before continuing.'
for gpu in GPUS:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

SEED = 42
tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print('TensorFlow:', tf.__version__, 'GPUs:', GPUS)

## 5. Configure one LOSO run

`TARGET_FOLD` is one-based in sorted subject order. Use a new `RUN_NAME` when changing architecture or training settings. Set an override to `None` to retain the value from the block-1 winner JSON.

In [ ]:
# Run identity and held-out subject.
RUN_NAME = 'block1_best_single_loso'
TARGET_FOLD = 1                 # One-based LOSO fold; exactly one target per run.
RESUME = True
ALLOW_SELECTION_LEAKAGE = False  # Required for targets used in block-1 NAS.

# Training knobs.
BATCH_SIZE = 128
MAX_EPOCHS = 100
PATIENCE = 15
MAX_PARAMETERS = 25_000_000
VERBOSE = 1

# Optional architecture overrides. None keeps the selected JSON value.
NUM_BLOCKS = None
CONV_REPEATS = None             # Example: '2,2,2,2'
WIDTH_MULTIPLIER = None         # 0.5, 1.0, or 2.0
TEMPORAL_KERNEL_SIZE = None     # 7, 11, or 15
DENSE_UNITS = None              # Example: '512' or '1024,512'
LEARNING_RATE = None
HEAD_TYPE = None                # 'flatten' or 'global_average'
CONVOLUTION_TYPE = None         # 'standard' or 'separable'
NORMALIZATION_TYPE = None       # 'batch', 'group', or 'layer'
POOLING_TYPE = None             # 'max' or 'average'
POOLING_SIZE = None             # 2 or 4

ARCHITECTURE_JSON = (
    PROJECT_DIR / 'data/cross_fitted_loso_new/blocks/block_001/search/best_architecture.json'
)
OUTPUT_BASE = Path('/content/drive/MyDrive/PainNAS') / RUN_NAME
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
print('Architecture:', ARCHITECTURE_JSON)
print('Output base:', OUTPUT_BASE)

## 6. Inspect the selected architecture

In [ ]:
import json
import pandas as pd
from IPython.display import display

winner = json.loads(ARCHITECTURE_JSON.read_text())
display(pd.DataFrame([winner['architecture']]).T.rename(columns={0: 'block-1 winner'}))
print('Inner subject accuracy:', winner['best_subject_accuracy_mean'])
print('Standard error:', winner['best_subject_accuracy_standard_error'])
print('Persisted parameters:', f"{winner['parameter_count']:,}")
print('Targets excluded from this NAS study:', winner['outer_block_subjects'])

## 7. Build runner arguments and audit the setup

In [ ]:
from scripts.run_painnas_best_single_loso import main as run_single_loso

RUN_ARGS = [
    '--data-dir', str(DATA_DIR),
    '--architecture-json', str(ARCHITECTURE_JSON),
    '--output-dir', str(OUTPUT_BASE),
    '--target-fold', str(TARGET_FOLD),
    '--seed', str(SEED),
    '--batch-size', str(BATCH_SIZE),
    '--max-epochs', str(MAX_EPOCHS),
    '--patience', str(PATIENCE),
    '--max-parameters', str(MAX_PARAMETERS),
    '--verbose', str(VERBOSE),
]
if RESUME:
    RUN_ARGS.append('--resume')
if ALLOW_SELECTION_LEAKAGE:
    RUN_ARGS.append('--allow-selection-leakage')

overrides = {
    '--num-blocks': NUM_BLOCKS,
    '--conv-repeats': CONV_REPEATS,
    '--width-multiplier': WIDTH_MULTIPLIER,
    '--temporal-kernel-size': TEMPORAL_KERNEL_SIZE,
    '--dense-units': DENSE_UNITS,
    '--learning-rate': LEARNING_RATE,
    '--head-type': HEAD_TYPE,
    '--convolution-type': CONVOLUTION_TYPE,
    '--normalization-type': NORMALIZATION_TYPE,
    '--pooling-type': POOLING_TYPE,
    '--pooling-size': POOLING_SIZE,
}
for flag, value in overrides.items():
    if value is not None:
        RUN_ARGS.extend([flag, str(value)])

print('Runner arguments:', ' '.join(RUN_ARGS))
run_single_loso([*RUN_ARGS, '--dry-run'])

## 8. Train and evaluate the single LOSO target

This cell initializes the selected architecture from scratch, trains on all non-target subjects, restores the best source-validation weights, and evaluates the held-out target once.

In [ ]:
run_single_loso(RUN_ARGS)

## 9. Inspect metrics and learning curves

In [ ]:
import matplotlib.pyplot as plt

target_roots = sorted(OUTPUT_BASE.glob(f'fold_{TARGET_FOLD:03d}_*'))
assert len(target_roots) == 1, f'Expected one target directory, found: {target_roots}'
TARGET_ROOT = target_roots[0]
RESULT_PATH = TARGET_ROOT / 'folds' / f'fold_{TARGET_FOLD:03d}.json'
result = json.loads(RESULT_PATH.read_text())

metric_names = ['accuracy', 'macro_f1', 'precision_t4', 'recall_t4', 'auroc', 'cross_entropy']
display(pd.DataFrame({name: [result['metrics'][name]] for name in metric_names}))
print('Target:', result['target_subject_key'])
print('Parameters:', f"{result['parameter_count']:,}")
print('Epochs:', result['epochs_ran'], 'Best epoch:', result['best_epoch'])
print('Result:', RESULT_PATH)

history = pd.DataFrame(result['history'])
display(history.tail())
ax = history[[c for c in ['loss', 'val_loss'] if c in history]].plot(title='Loss')
ax.set_xlabel('Epoch')
plt.show()
ax = history[[c for c in ['accuracy', 'val_accuracy', 'macro_f1', 'val_macro_f1'] if c in history]].plot(
    title='Training and source-validation metrics'
)
ax.set_xlabel('Epoch')
ax.set_ylim(0, 1)
plt.show()

## 10. Optional runtime cleanup

In [ ]:
import gc
tf.keras.backend.clear_session()
gc.collect()
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print('Cleanup complete')